# report

> `ComparisonResultList` -- the list-with-reporting-helpers that `compare()` returns.

`compare()` (built in `03_compare`, the next notebook) gives us back a list
of `ComparisonResult` objects, one per engine. A plain list can already do
the basics -- we can loop over it, index into it, check its length -- but on
its own it can't answer the two questions we actually care about once a
comparison finishes: "put this side by side for me to read" and "show me
exactly what differs between two of these engines." `ComparisonResultList`
is a list that also knows how to answer those two questions, so we never
have to write that formatting code ourselves each time we run a comparison.

In [ ]:
#| default_exp report

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

from estravon_bench.io import ComparisonResult

## `ComparisonResultList`

We define this as a subclass of Python's built-in `list` rather than a
separate wrapper object holding a list inside it. That choice is deliberate:
it means `compare()` can hand us back something that behaves exactly like a
normal list everywhere we'd expect one to (`for r in results:`,
`results[0]`, `len(results)`, `list(results)`) while also carrying the three
extra methods below. We never have to "unwrap" it to get at our results.

- **`to_markdown_table()`** turns the whole list into one side-by-side table
  -- engine, time, cost, whether it ran locally, page count, and status.
  This is the everyday output of a comparison: read down the columns and we
  can already see which engine was fastest, which was free, and which
  failed, without opening anything else. We love markdown.
- **`get(engine)`** finds one specific engine's result by name, or gives us
  `None` if that engine isn't in the list -- a small convenience so we don't
  have to write our own search loop when we want to look at just one
  engine's output in detail.
- **`diff(engine_a, engine_b)`** shows us, line by line, exactly where two
  engines' Markdown output disagrees -- the same format `git diff` or
  `diff -u` produce. A table tells us *that* two engines differ in timing or
  cost; `diff()` is how we actually see *what* one engine wrote differently
  from another on the same page.

In [ ]:
#| export
class ComparisonResultList(list):
    """list[ComparisonResult] with side-by-side reporting helpers.

    compare() returns this type directly -- callers can treat it as a plain
    list (iterate, index, len()) or call the extra methods below.
    """

    def to_markdown_table(self) -> str:
        """Side-by-side engine | time | cost | local? | pages | status table."""
        header = "| engine | time (s) | cost (usd) | local | pages | status |\n"
        header += "|---|---|---|---|---|---|\n"
        rows = []
        for r in self:
            if not r.ok:
                rows.append(f"| {r.engine} | - | - | - | - | error: {r.error} |")
                continue
            time_s = f"{r.predict_time_s:.2f}" if r.predict_time_s is not None else "-"
            cost = "free (local)" if r.local else (f"${r.cost_usd:.4f}" if r.cost_usd is not None else "-")
            pages = r.page_count if r.page_count is not None else "-"
            rows.append(f"| {r.engine} | {time_s} | {cost} | {'yes' if r.local else 'no'} | {pages} | ok |")
        return header + "\n".join(rows)

    def get(self, engine: str) -> ComparisonResult | None:
        for r in self:
            if r.engine == engine:
                return r
        return None

    def diff(self, engine_a: str, engine_b: str) -> str:
        """Unified line diff of two engines' Markdown output, for eyeballing."""
        import difflib
        a, b = self.get(engine_a), self.get(engine_b)
        if a is None or b is None:
            missing = engine_a if a is None else engine_b
            raise KeyError(f"no result for engine {missing!r}")
        if not a.ok or not b.ok:
            raise ValueError("cannot diff an engine that errored")
        a_lines = (a.markdown or "").splitlines(keepends=True)
        b_lines = (b.markdown or "").splitlines(keepends=True)
        return "".join(difflib.unified_diff(a_lines, b_lines, fromfile=engine_a, tofile=engine_b))

### Try it

We build a small `ComparisonResultList` by hand below -- one engine that
succeeded (`mineru`, free and local) and one that failed (`mistral`, no API
key) -- the same shape `compare()` would hand us after a real run. Printing
`to_markdown_table()` shows us the successful row and the failed row
rendered side by side, including the "free (local)" cost label rather than
a bare `0.0`. Then we deliberately try to `diff()` the failed engine against
the working one, to see that it refuses with a clear error instead of
crashing on missing Markdown -- there's nothing sensible to diff when one
side never produced any text.

In [ ]:
#| hide
lst = ComparisonResultList([
    ComparisonResult(engine="mineru", markdown="# Hi\nfoo", cost_usd=0.0, local=True, page_count=3, predict_time_s=12.0),
    ComparisonResult(engine="mistral", error="no API key"),
])
table = lst.to_markdown_table()
assert "mineru" in table and "free (local)" in table
assert "error: no API key" in table
assert lst.get("mineru").page_count == 3
assert lst.get("nope") is None
print(table)
try:
    lst.diff("mineru", "mistral")
    raise AssertionError("should have raised")
except ValueError as exc:
    print(f"diff() correctly raised: {exc}")

---
Next: [`03_compare`](03_compare.ipynb) -- `compare()` itself, which builds
the `ComparisonResultList` we've just been exploring, using the `Client`
and `LocalEngineProcess` from `01_client`.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()